In [ ]:
# Default parameters for manual testing (ADF will overwrite these at runtime)
entity_name = "ALL"          # Options: "ALL", "suppliers", "purchase_orders", "deliveries", "inventory"
processing_date = "2026-07-22"

In [ ]:
# ------------------------------------------------------------------
# Notebook: DQ_Silver (Supports Single Entity or "ALL") + Logging
# ------------------------------------------------------------------
from pyspark.sql.functions import col
import csv, os, io

# Storage configuration
storage_account = "stlakesupply"
container = "medallion"

# Extract date partition parts from parameter
year, month, day = processing_date.split("-")

# Entity primary key definitions
key_map = {
    "suppliers": ["supplier_id"],
    "purchase_orders": ["order_id"],
    "deliveries": ["delivery_id"],
    "inventory": ["warehouse_id", "product_id"]
}

# Determine entities to validate
if entity_name.upper() == "ALL":
    entities_to_check = list(key_map.keys())
else:
    if entity_name not in key_map:
        raise ValueError(f"Unknown entity '{entity_name}'. Must be one of {list(key_map.keys())} or 'ALL'.")
    entities_to_check = [entity_name]

print(f"--- Starting Silver Data Quality Checks for Date: {processing_date} ---")
print(f"Target entity parameter: {entity_name}")
print(f"Entities queued ({len(entities_to_check)}): {', '.join(entities_to_check)}\n")

# Control CSV path for logging
control_csv_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/control/quality_gate_status.csv"

def append_quality_log(entity, date, status, details):
    """Append a row to the quality gate control CSV in ADLS."""
    # Use Hadoop FileSystem API for reliable append
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    sc = spark.sparkContext
    URI = sc._gateway.jvm.java.net.URI
    Path = sc._gateway.jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._gateway.jvm.org.apache.hadoop.fs.FileSystem
    conf = sc._jsc.hadoopConfiguration()
    fs = FileSystem.get(URI(control_csv_path), conf)

    # If file doesn't exist, create with header
    if not fs.exists(Path(control_csv_path)):
        out_stream = fs.create(Path(control_csv_path))
        out_stream.write("entity_name,processing_date,status,details\n".encode())
    else:
        out_stream = fs.append(Path(control_csv_path))

    line = f"{entity},{date},{status},{details}\n"
    out_stream.write(line.encode())
    out_stream.close()

failed_entities = {}

# ------------------------------------------------------------------
# Iterative Quality Check Execution
# ------------------------------------------------------------------
for current_entity in entities_to_check:
    print("=" * 60)
    print(f" Quality Checks: {current_entity}")
    print("=" * 60)

    silver_partition = (
        f"abfss://{container}@{storage_account}.dfs.core.windows.net/silver/"
        f"{current_entity}/year={year}/month={month}/day={day}/"
    )
    key_cols = key_map[current_entity]

    try:
        # Load Silver Parquet Partition
        df = spark.read.parquet(silver_partition)
        total_rows = df.count()
        print(f"  [+] Loaded Parquet partition ({total_rows} total rows)")

        # 1. Check Row Count > 0
        if total_rows == 0:
            raise Exception(f"Partition is empty (0 rows) for date {processing_date}")

        # 2. Check for Null Primary Keys
        for k in key_cols:
            null_count = df.filter(col(k).isNull()).count()
            if null_count > 0:
                raise Exception(f"Found {null_count} null value(s) in key column '{k}'")

        # 3. Check Key Uniqueness (Duplicates)
        distinct_count = df.select(key_cols).distinct().count()
        if distinct_count != total_rows:
            raise Exception(f"Duplicate keys detected ({distinct_count} distinct key combinations vs {total_rows} rows)")

        print(f"  [✓] All DQ checks passed for '{current_entity}'!\n")
        # Log success
        append_quality_log(current_entity, processing_date, "PASS", "All checks passed")

    except Exception as e:
        err_msg = str(e)
        print(f"  [✗] DQ FAIL for '{current_entity}': {err_msg}\n")
        failed_entities[current_entity] = err_msg
        # Log failure
        append_quality_log(current_entity, processing_date, "FAIL", err_msg)

# ------------------------------------------------------------------
# Execution Summary & Pipeline Assertion
# ------------------------------------------------------------------
print("=" * 60)
if failed_entities:
    summary_lines = [f"  - {e}: {msg}" for e, msg in failed_entities.items()]
    summary_text = "\n".join(summary_lines)
    raise Exception(f"Data Quality verification failed for {len(failed_entities)} entity/entities:\n{summary_text}")
else:
    print("SUCCESS: All Silver Data Quality checks passed!")